In [ ]:
import pandas as pd
import os
import re
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import json
import matplotlib.patheffects as pe
import networkx as nx

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# Get VAFs for each clusters 

In [ ]:
donor = 'PD60967'
tumour = 'PD60967'

In [ ]:
## Load pyclone results for hierarchy analysis

node_mutations = {}
wd = '../data/LCM_analysis/'


pyclone_input = pd.read_csv(wd+'pyclone_vi/{}_input.tsv'.format(donor), sep = '\t')
samples = sorted(list(set(pyclone_input['sample_id'])))
pyclone_input['VAF'] = pyclone_input['alt_counts']/(pyclone_input['alt_counts']+pyclone_input['ref_counts'])
pyclone_input['VAF_corrected'] = np.where(pyclone_input['normal_cn']==1, 
                                            pyclone_input['VAF']/2, pyclone_input['VAF'])


# Load pyclone output
pyclone_output = pd.read_csv(wd+'pyclone_vi/{}_output.tsv'.format(donor), sep = '\t')
pyclone_output = pyclone_output[pyclone_output['cluster_assignment_prob']>0.95] # remove low prob variants

    
for cluster in sorted(list(set(pyclone_output['cluster_id']))):
        mutations = sorted(list(set(set(pyclone_output[pyclone_output['cluster_id']==cluster]['mutation_id']))))
        mut_count = len(mutations)
        node_mutations[cluster+1] = mut_count

In [ ]:
node_mutations_log = {k: np.log(v) for k, v in node_mutations.items()}

In [ ]:
# mutation filter:

## Load pyclone results for hierarchy analysis

all_median_VAFs = {}
all_cluster_VAFs = {}
wd = '../data/LCM_analysis/'

mutation_clusters = {}
for cluster in sorted(list(set(pyclone_output['cluster_id']))):
        mutations = sorted(list(set(set(pyclone_output[pyclone_output['cluster_id']==cluster]['mutation_id']))))
        mut_count = len(mutations)
        mutation_clusters[cluster] = {}
        mutation_clusters[cluster]['count'] = mut_count
        mutation_clusters[cluster]['mutations'] = mutations

    
# Generate VAF plot
median_VAF_df = []
all_VAF_df = []
for sample in sorted(list(set(pyclone_input['sample_id']))):
        for cluster in mutation_clusters.keys():
            cluster_df = pyclone_input[
                (pyclone_input['mutation_id'].isin(mutation_clusters[cluster]['mutations'])) &
                (pyclone_input['sample_id']==sample)
                ].reset_index(drop=True)
            cluster_df['cluster_id'] = cluster
            all_VAF_df.append(cluster_df)

            VAF = cluster_df['VAF_corrected'].median()
            median_VAF_df.append(
                pd.DataFrame({'sample':sample,
                        'cluster':cluster,
                        'VAF':VAF}, index = [0])
            )

all_VAF_df = pd.concat(all_VAF_df, ignore_index=True)

median_VAF_df = pd.concat(median_VAF_df, ignore_index=True)\
        .pivot(index = 'cluster', columns = 'sample', values = 'VAF')



# Phylogeny and distance

In [ ]:
with open(wd + '/pyclone_vi/' + tumour + ".json", "r") as f:
    data = json.load(f)

parents = data['parents']

In [ ]:
# number of nodes (0 to 13 inclusive)
n_nodes = len(parents)+1

# get colormap
cmap = plt.get_cmap("tab20c")

# assign colors in order
node_colors = {
    node: cmap(node / (n_nodes - 1))  # normalize to [0,1]
    for node in range(n_nodes)
}

In [ ]:
from collections import defaultdict
import pandas as pd

def pairtree_pairwise_distances(parents, node_mutations=None, root=0, include_root=False):
    
    node_mutations = node_mutations or {}
    parents = [int(x) for x in parents]

    n_nodes = len(parents) + 1
    all_nodes = list(range(n_nodes))

    # Build parent map and children map
    parent_of = {root: None}
    children = defaultdict(list)

    for i, p in enumerate(parents):
        node = i + 1
        parent_of[node] = p
        children[p].append(node)

    # Branch lengths
    branch_len = {node: float(node_mutations.get(node, 0)) for node in all_nodes}
    branch_len[root] = 0.0

    # Distance from root
    depth = {}
    dist_from_root = {}

    def dfs(node, d=0, cumdist=0.0):
        depth[node] = d
        dist_from_root[node] = cumdist
        for ch in children[node]:
            dfs(ch, d + 1, cumdist + branch_len[ch])

    dfs(root)

    # Ancestor chain helper
    def ancestors(node):
        out = []
        while node is not None:
            out.append(node)
            node = parent_of[node]
        return out

    # LCA helper
    def lca(a, b):
        anc_a = set(ancestors(a))
        cur = b
        while cur is not None:
            if cur in anc_a:
                return cur
            cur = parent_of[cur]
        return None

    # Which nodes to include
    nodes = all_nodes if include_root else [n for n in all_nodes if n != root]

    # Pairwise distances
    mat = []
    for a in nodes:
        row = []
        for b in nodes:
            ab_lca = lca(a, b)
            d = dist_from_root[a] + dist_from_root[b] - 2 * dist_from_root[ab_lca]
            row.append(d)
        mat.append(row)

    dist_df = pd.DataFrame(mat, index=nodes, columns=nodes)
    return dist_df, dist_from_root

import matplotlib.pyplot as plt
from collections import defaultdict

def plot_pairtree_phylogeny(
    parents,
    node_mutations=None,
    plot_branch_lengths=None,
    root=0,
    node_colors = None,
    figsize=(10, 6),
    label_internal=True,
    label_branch_lengths=True,
    path=None
):

    node_mutations = node_mutations or {}
    plot_branch_lengths = plot_branch_lengths or node_mutations

    parents = [int(x) for x in parents]

    n_nodes = len(parents) + 1
    all_nodes = list(range(n_nodes))

    children = defaultdict(list)
    parent_of = {root: None}

    for i, parent in enumerate(parents):
        node = i + 1
        if parent not in all_nodes:
            raise ValueError(
                f"Invalid parent ID {parent} for node {node}. "
                f"Expected parent in 0..{n_nodes-1}."
            )
        parent_of[node] = parent
        children[parent].append(node)

    visited = set()
    stack = set()

    def dfs_check(node):
        if node in stack:
            raise ValueError(f"Cycle detected involving node {node}.")
        if node in visited:
            return
        stack.add(node)
        visited.add(node)
        for ch in children[node]:
            dfs_check(ch)
        stack.remove(node)

    dfs_check(root)

    if len(visited) != n_nodes:
        missing = sorted(set(all_nodes) - visited)
        raise ValueError(f"These nodes are not reachable from root {root}: {missing}")

    # Actual lengths for labels
    actual_len = {node: float(node_mutations.get(node, 1)) for node in all_nodes}
    actual_len[root] = 0.0

    # Plot lengths for geometry
    plot_len = {node: float(plot_branch_lengths.get(node, actual_len[node])) for node in all_nodes}
    plot_len[root] = 0.0

    # x = cumulative plotted distance from root
    x = {}

    def assign_x(node, cur_x):
        x[node] = cur_x
        for ch in children[node]:
            assign_x(ch, cur_x + plot_len[ch])

    assign_x(root, 0.0)

    # y = tips spaced evenly; internal nodes centered
    y = {}
    leaf_order = []

    def assign_y(node):
        if len(children[node]) == 0:
            y[node] = len(leaf_order)
            leaf_order.append(node)
        else:
            for ch in children[node]:
                assign_y(ch)
            y[node] = sum(y[ch] for ch in children[node]) / len(children[node])

    assign_y(root)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)

    for parent, child_list in children.items():
        if not child_list:
            continue

        ys = [y[ch] for ch in child_list]

        # vertical connector
        ax.plot([x[parent], x[parent]], [min(ys), max(ys)], lw=1.5, color = 'black')

        # horizontal branches
        for ch in child_list:
            ax.plot([x[parent], x[ch]], [y[ch], y[ch]], lw=1.5, color = 'black')

    # Nodes and labels
    for node in all_nodes:
        if node_colors!=None:
            node_color = node_colors[node]
        else:
            node_color = 'skyblue'
        
        ax.plot(x[node], y[node], "o", ms=20, color = node_color)

        is_leaf = len(children[node]) == 0
        if node == root or is_leaf or label_internal:
            label = str(node)
            if label_branch_lengths and node != root:
                label += f" [{actual_len[node]:g}]"
            ax.text(x[node] + 0.7, y[node], label, va="center", fontsize=12,
                   bbox=dict(facecolor="white", edgecolor="none", pad=1))

    ax.set_xlabel("Mutational distance")
    ax.set_ylabel("")
    ax.set_title(f"Phylogeny (root={root})")
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_yticks([])                 # remove y tick marks + labels
    ax.spines["left"].set_visible(False)   # remove left axis line
    
    plt.tight_layout()
    if path==None:
        plt.show()
    else:
        plt.savefig(path)

In [ ]:
plot_pairtree_phylogeny(parents, node_mutations, node_mutations_log, 
                        node_colors = node_colors, figsize = (8, 4),
                       path = '../plots/tree_with_distance_log.pdf')


In [ ]:
plot_pairtree_phylogeny(parents, node_mutations, node_mutations, 
                        label_branch_lengths = False,
                        node_colors = node_colors, figsize = (10, 4),
                       path = '../plots/tree_with_distance.pdf')

In [ ]:
dist_df, dist_from_root = pairtree_pairwise_distances(parents, node_mutations, root=0)

# Apply Rao's Q

In [ ]:
# check low-frac and problem clusters:
median_VAF_df.max(axis=1)

In [ ]:
with open(wd+'pyclone_vi/{}.json'.format(donor)) as f:
    pairtree_output = json.load(f)
tree_bound_freqs = pd.DataFrame(pairtree_output['eta'], columns = pairtree_output['samples']).transpose()

In [ ]:
def rao_q(p, D):
    
    p = p.fillna(0).astype(float)

    if p.sum() == 0:
        return float("nan")

    p = p / p.sum()

    # align matrix
    D = D.loc[p.index, p.index]

    return float(p.values @ D.values @ p.values)

def compute_rao_all(fractions_df, dist_df):
    
    results = {}

    for sample in fractions_df.columns:
        p = fractions_df[sample]
        results[sample] = rao_q(p, dist_df)

    return pd.Series(results, name="rao_q")

# remove root and bad clones (clade with extremely low VAFs in sample);
remove = [0, 1, 2, 10]

tree_bound_freqs = tree_bound_freqs[[i for i in tree_bound_freqs.columns if i not in remove]]
tree_bound_freqs = tree_bound_freqs.divide(tree_bound_freqs.sum(axis=1), axis=0)

dist_df = dist_df[[i for i in dist_df.columns if i not in remove]]
dist_df = dist_df.T[[i for i in dist_df.columns if i not in remove]]

Q = compute_rao_all(tree_bound_freqs.T, dist_df).sort_values()

In [ ]:
# get clone clusters
pt_cluster_ids = pd.read_csv('../data/LCM_ST_comparison/AT3/AT3_PT_labels.csv', index_col=0)

In [ ]:
fig, ax = plt.subplots(figsize=(0.7, 8))  # wider than tall for LR trees

sns.heatmap(pd.DataFrame(Q.loc[pt_cluster_ids['PDID'].iloc[::-1]]), cmap = 'Reds',
           linewidths=0.5, linecolor="black")

plt.savefig('../plots/raos_Q.pdf')

In [ ]:
print("{}/{} = {}".format(len(Q[Q>300]), len(Q), len(Q[Q>300])/len(Q)))

In [ ]:
print("{}/{} = {}".format(len(Q[Q<100]), len(Q), len(Q[Q<100])/len(Q)))

In [ ]:
sns.histplot(Q, binwidth = 20)

In [ ]:
dist_from_root_sorted = dict(sorted(dist_from_root.items(), key=lambda x: x[1]))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 8))  # wider than tall for LR trees

plot_df = tree_bound_freqs.T[pt_cluster_ids['PDID'].iloc[::-1]].T
plot_df = plot_df[[i for i in dist_from_root_sorted if i in plot_df.columns]]

sns.heatmap(plot_df, cmap = 'Reds', linewidths=0.5, linecolor="black")

plt.savefig('../plots/tree_bound_freq.pdf')

In [ ]:
pd.DataFrame(Q.loc[pt_cluster_ids['PDID'].iloc[::-1]]).to_csv('../data/raos_Q.csv')